# 4 Datenobjekte abholen


Dieses Script erstellt in 'object' die Unterordner mit dem Identifier, die Files werden ohne weitere Zwischenverarbeitung oder Prüfung hierhin kopiert. 
Das Migrieren sowie Entzippen erfolgt vorerst manuell. 



## Variante A: Datenobjekte liegen auf einem Laufwerk

Gültig für Digitalisate aus der Sosa, bzw. Objekte, die auf lokalen Laufwerken liegen. 
Achtung! Dieses Notebook kann in der Ausführung sehr lange dauern, da die Dateien riesig sind. 
Voraussetzung: mit VPN verbunden / im unilu-Netzwerk, mit Laufwerk G verbunden.

Aufgrund der geringen Menge und der diversen Metadatenquellen werden diese Zipkapseln von Hand vom Laufwerk G auf die Workbench verschoben. Sie werden alle manuell entzippt. 

Die Zipkapseln sind alle nach folgender Struktur benannt:

    DOI/ID _ (Digitalisierungsdatum/Workflow?) _ master _ version . zip

Beispiele:

    000190118_20150320T000256_master_ver1.zip
    10_7891_e-manuscripta-108732.zip

Die Objekte liegen auf G:\ZHB-Sosa_Digital\digital. Die vollständigen Pfade auf G sind in der Eingabedatei ergänzt und ist in die Infojson unter 'additional' abgelegt. 
Der Dateipfad auf der Workbench wird nach dem DOI benannt. 



In [1]:
import os
import config
import json
from datetime import datetime
import zipfile
import shutil
from pathlib import Path

localdrive = config.user_root
org_id = config.organisation_id
file_name = f'{config.inventory_file}'
objects_path = f'{localdrive}/{config.collection_id}/{config.object_path}'
counter = 0

with open(file_name) as data_file:    
    data = json.load(data_file)
    for value in data:
        counter += 1
        
        # get path to G drive:
        g_path = value["additional"]
        #print(f"Origin path: {g_path}")
        
        # create new object folder name (AIP path). The full path is needed here for copying the files.
        #foldername = value["signature"][(len(org_id)+1):]
        foldername = value["references"][-1]
        aip_path = f"{objects_path}/{foldername}"
         
        print("Destination path:",aip_path)
                
        # prepare object folder: make a directory for each object
        Path(f'{aip_path}').mkdir(parents=True, exist_ok=True)
        
        
        filenames = []
        # Iterate directory, check if current file_path is a file
        try:
            for file_path in os.listdir(g_path):
                print("Origin files:", file_path)
                if os.path.isfile(os.path.join(g_path, file_path)):
                    filenames.append(file_path)
                else:
                    print("---------------- not a file!-----------------------")
        except FileNotFoundError:
            print(f"The directory {g_path} does not exist")
        except PermissionError:
            print(f"Permission denied to access the directory {g_path}")
        except OSError as e:
            print(f"An OS error occurred: {e}")
        
        for file in filenames:   
            aip_file = Path(aip_path+'/'+file)
            #print("File to be copied:",aip_path+'/'+file)
            if aip_file.exists():
                # path exists
                print("*** Path exists, file already copied")
            else:
                # copy file:
                print("Copying file:", file)
                print("Time started copying:",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))
                shutil.copy(g_path+'/'+file, aip_path+'/'+file) 
            
                print("File copied successfully.")
                
        #debugging:        
        if counter == 1:
            break
        

            
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))

Destination path: C:/Users/HeimK/switchdrive/jupyter/dlza/sosa_emanus/objects/10_7891_e-manuscripta-108732
Origin files: 10_7891_e-manuscripta-108732
---------------- not a file!-----------------------
Origin files: 10_7891_e-manuscripta-108732.zip
Copying file: 10_7891_e-manuscripta-108732.zip
Time started copying: 2024-01-05 17:14:01
File copied successfully.
Finished at  2024-01-05 17:14:03


## Variante B: Download aus Zenodo

Die Datenabholung für Zenodo-Repositories funktioniert etwas anders: mittels HTTP download direkt von zenodo.

Test-Daten, Stand Januar 2024:
counter 60-70: enthält ein Datenset (page 7)

TODO: check different file types, check errors. 


In [5]:
import fitz  # PyMuPDF
import requests
import json
import os
import time
import config
from datetime import datetime
from pathlib import Path




file_name = config.inventory_file
community = config.collection_id
objects_path = f'{config.user_root}/{config.collection_id}/{config.object_path}'
download_manually = f'{community}_download_manually.txt'

counter = 0
ACCESS_TOKEN = config.access_token

with open(file_name, encoding="utf-8") as data_file:    
    data = json.load(data_file)
    for value in data:
        
        counter = counter+1
        
        if counter<= 75: # debug mode, for prod: replace with if True:
            
            # find necessary values  
            foldername = value["references"][1]
            aip_path = f"{objects_path}/{foldername}"         

            identifiers = {}
            for item in value["identifiers"]:
                # split identifiers in dict
                [key, value] = item.split(':',1)
                identifiers[key] = value

            zenodo_id = identifiers['zenodo']


            # prepare object folder: make a directory for each object
            Path(f'{aip_path}').mkdir(parents=True, exist_ok=True)

            # get the file download link from zenodo:        
            zenodo_link = f'https://zenodo.org/api/records/{zenodo_id}/files'       
            print(f"\n#{counter}: Start downloading files from:",zenodo_link)
            response = requests.get(zenodo_link, params={'access_token': ACCESS_TOKEN})
            file_object = response.json()    

            try: 
                for entry in file_object['entries']:
                    download_url = entry['links']['content']
                    file_name = entry['key']
                    mimetype = entry['mimetype']

                    print("filename:",file_name, "mimetype:",mimetype)
                    print(download_url)
                    local_file = f'{aip_path}/{file_name}'

                    # distinguish between pdf files and other files

                    if mimetype == 'application/pdf':

                        response = requests.get(download_url)
                        with open(local_file, mode="wb") as file:
                            file.write(response.content)
                            print("Downloaded:",aip_path)
                            # wait 1 second for every record so as not to overshoot zenodo rate limiting. 
                            time.sleep(1) 
                            # check downloaded files for integrity:
                            try:
                                with fitz.open(local_file) as pdf_document:
                                    if pdf_document.page_count != 0:
                                        print("PDF page count:",pdf_document.page_count)
                            except Exception as e:
                                print(f"---   Error checking PDF file {file_name}: {e}")
                                # append download url to download_manually.txt
                                with open(download_manually, 'a') as file:
                                    file.write(download_url)
                                    file.write("\n")
                                    print(f"URL {download_url} appended to {download_manually}\n")                                
                        
                    else:
                        print("---   Not a PDF, download manually:")
                        # append download url to download_manually.txt
                        with open(download_manually, 'a') as file:
                            file.write(download_url)
                            file.write("\n")
                            print(f"URL {download_url} appended to {download_manually}\n")
            except KeyError:
                print("---   KeyError: files not found, download manually:")
                # append download url to download_manually.txt
                with open(download_manually, 'a') as file:
                    file.write(zenodo_link)
                    file.write("\n")
                    print(f"URL {zenodo_link} appended to {download_manually}\n")
                    

            

print("\nFinished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))


#1: Start downloading files from: https://zenodo.org/api/records/10466776/files
filename: Cache_WareReinheit.pdf mimetype: application/pdf
https://zenodo.org/api/records/10466776/files/Cache_WareReinheit.pdf/content
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10466776
PDF page count: 92

#2: Start downloading files from: https://zenodo.org/api/records/10471407/files
filename: BRAVEpapers_AM.pdf mimetype: application/pdf
https://zenodo.org/api/records/10471407/files/BRAVEpapers_AM.pdf/content
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10471407
---   Error checking PDF file BRAVEpapers_AM.pdf: cannot open empty document
URL https://zenodo.org/api/records/10471407/files/BRAVEpapers_AM.pdf/content appended to lory_unilu_download_manually.txt


#3: Start downloading files from: https://zenodo.org/api/records/10471212/files
filename: e077454.full.pdf mimetype: application/pdf
https://zenodo.org/api/records/

PDF page count: 6

#22: Start downloading files from: https://zenodo.org/api/records/10400900/files
filename: Holderlins_Tempo.pdf mimetype: application/pdf
https://zenodo.org/api/records/10400900/files/Holderlins_Tempo.pdf/content
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10400900
PDF page count: 22

#23: Start downloading files from: https://zenodo.org/api/records/10365584/files
filename: Arni_Sommer_Teuscher_Diagrammatik_der_Verwandtschaft_PV.pdf mimetype: application/pdf
https://zenodo.org/api/records/10365584/files/Arni_Sommer_Teuscher_Diagrammatik_der_Verwandtschaft_PV.pdf/content
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10365584
---   Error checking PDF file Arni_Sommer_Teuscher_Diagrammatik_der_Verwandtschaft_PV.pdf: cannot open empty document
URL https://zenodo.org/api/records/10365584/files/Arni_Sommer_Teuscher_Diagrammatik_der_Verwandtschaft_PV.pdf/content appended to lory_unilu_download

PDF page count: 8

#43: Start downloading files from: https://zenodo.org/api/records/10160976/files
filename: 1-s2.0-S0939475323002867-main.pdf mimetype: application/pdf
https://zenodo.org/api/records/10160976/files/1-s2.0-S0939475323002867-main.pdf/content
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10160976
PDF page count: 13

#44: Start downloading files from: https://zenodo.org/api/records/10160623/files
filename: Previsic_Interview_Bull23_Previsic_GzD_Holozaen_2023.pdf mimetype: application/pdf
https://zenodo.org/api/records/10160623/files/Previsic_Interview_Bull23_Previsic_GzD_Holozaen_2023.pdf/content
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_10160623
PDF page count: 6

#45: Start downloading files from: https://zenodo.org/api/records/10160392/files
filename: 231107_DIEBOLD_Freizuegigkeit.pdf mimetype: application/pdf
https://zenodo.org/api/records/10160392/files/231107_DIEBOLD_Freizuegigkeit.p

Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_8403906
PDF page count: 93

#63: Start downloading files from: https://zenodo.org/api/records/8392075/files
filename: Skript_Rechtsfragen_Palliative_Care_2019-14.pdf mimetype: application/pdf
https://zenodo.org/api/records/8392075/files/Skript_Rechtsfragen_Palliative_Care_2019-14.pdf/content
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_8392075
PDF page count: 58

#64: Start downloading files from: https://zenodo.org/api/records/8389747/files
filename: philosophies-08-00062.pdf mimetype: application/pdf
https://zenodo.org/api/records/8389747/files/philosophies-08-00062.pdf/content
Downloaded: C:/Users/HeimK/switchdrive/jupyter/dlza/lory_unilu/objects/10_5281_zenodo_8389747
PDF page count: 16

#65: Start downloading files from: https://zenodo.org/api/records/8359296/files
filename: Reibungsgewinne_-_Reibungsverluste_(2023)_Gesamtbuch(1).pdf mimetype: application/